In [4]:
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import rasterio
import xarray as xr
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import boto3

import os
os.environ['AWS_PROFILE'] = 'ctrees'
os.environ['GDAL_DISABLE_READDIR_ON_OPEN'] = 'YES'
os.environ['VSI_CURL_ALLOW_CREDENTIALS'] = 'YES'


In [ ]:
feature_names = [f'emb_{i:03d}' for i in range(128)]

In [6]:
# get all s3 files in training_parquet_prefix
s3 = boto3.client('s3')
bucket_name = 'ctrees-development'
prefix = 'activity/tessera/logging_pantropical_v1/extracted_embeddings/'
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.parquet')]
print(files)

# load all parquet files into a single dataframe
dfs = []
for file in files:
    path = f's3://{bucket_name}/{file}'
    df = pd.read_parquet(path)
    dfs.append(df)
df = pd.concat(dfs, ignore_index=True)

['activity/tessera/logging_pantropical_v1/extracted_embeddings/embeddings_PAGE_0602-0978.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings/embeddings_PAGE_0625-0976.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings/embeddings_PAGE_0633-0967.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings/embeddings_PAGE_0666-0974.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings/embeddings_PAGE_0667-0973.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings/embeddings_PAGE_0670-0972.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings/embeddings_PAGE_0678-0970.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings/embeddings_PAGE_0690-1008.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings/embeddings_PAGE_0691-1051.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings/embeddings_PAGE_0696-0967.parquet', 'activity/tessera/l

In [ ]:
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Split with stratification
X_train, X_test = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df['class']
)

# Balance training set: undersample class 0 to match class 1
n_class1_train = (X_train['class'] == 1).sum()
class0_train_idx = X_train[X_train['class'] == 0].sample(n=n_class1_train, random_state=42).index
class1_train_idx = X_train[X_train['class'] == 1].index
X_train = X_train.loc[class0_train_idx.union(class1_train_idx)].sample(frac=1, random_state=42)

print(f"Training set after balancing: class 0 = {(X_train['class']==0).sum()}, class 1 = {(X_train['class']==1).sum()}")
print(f"Test set (original distribution): class 0 = {(X_test['class']==0).sum()}, class 1 = {(X_test['class']==1).sum()}")

train_data = lgb.Dataset(X_train[feature_names], label=X_train['class'])
test_data = lgb.Dataset(X_test[feature_names], label=X_test['class'], reference=train_data)

param_grid = {
    'num_leaves': [31, 63, 127],
    'min_child_samples': [10, 20, 50],
}

best_score = float('inf')
best_params = None
best_num_boost_round = None

for grid_params in ParameterGrid(param_grid):
    params = {
        'objective': 'binary',
        'metric': ['binary_logloss', 'auc'],
        'verbosity': -1,
        'bagging_freq': 1,
        'force_col_wise': True,
        'learning_rate': 0.1,
        **grid_params
    }

    cv_results = lgb.cv(
        params,
        train_data,
        nfold=3,
        num_boost_round=500,
        stratified=True,
        seed=42,
        callbacks=[lgb.early_stopping(30)]
    )

    mean_score = min(cv_results['valid binary_logloss-mean'])
    best_iter = np.argmin(cv_results['valid binary_logloss-mean']) + 1

    if mean_score < best_score:
        best_score = mean_score
        best_params = params
        best_num_boost_round = best_iter

print("Best params:", best_params)
print("Best CV logloss:", best_score)
print("Best num_boost_round:", best_num_boost_round)

bst = lgb.train(
    best_params,
    train_data,
    num_boost_round=best_num_boost_round,
    valid_sets=[test_data],
    callbacks=[lgb.early_stopping(50)]
)

y_pred_prob = bst.predict(X_test[feature_names], num_iteration=bst.best_iteration)
y_pred = (y_pred_prob >= 0.5).astype(int)

accuracy = accuracy_score(X_test['class'], y_pred)
auc = roc_auc_score(X_test['class'], y_pred_prob)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'AUC: {auc:.4f}')
print(classification_report(X_test['class'], y_pred))

Training set after balancing: class 0 = 595000, class 1 = 595000
Test set (original distribution): class 0 = 105000, class 1 = 105000
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[500]	valid's binary_logloss: 0.511985 + 0.000637247	valid's auc: 0.82823 + 0.000729514
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[500]	valid's binary_logloss: 0.488397 + 0.00106317	valid's auc: 0.846267 + 0.000952773
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[500]	valid's binary_logloss: 0.468003 + 0.000905599	valid's auc: 0.859956 + 0.00069627
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[500]	valid's binary_logloss: 0.511905 + 0.000741477	valid's auc: 0.82828 + 0.000730903
Training until validation scores don't improve for 30 rounds
Did not meet earl

In [16]:
model_name = 'pantropical_v1.pkl'
joblib.dump(bst, model_name)

# save in s3
s3 = boto3.client('s3')
bucket_name = 'ctrees-development'
object_name = f'activity/tessera/logging_pantropical_v1/model/{model_name}'
s3.upload_file(model_name, bucket_name, object_name)